In [ ]:
#%pip install pandas numpy
#%pip install pathlib

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [26]:
import pandas as pd
import numpy as np

df=pd.read_csv("secop_ii_agencia_logistica_limpio.csv")

cols_fecha=[c for c in df.columns if c.startswith("fecha") or c=="ultima_actualizacion"]
for c in cols_fecha:
    df[c]=pd.to_datetime(df[c], errors="coerce")
print(df.shape)

print("Dimensiones del archivo:", df.shape)
print("Columnas de fecha convertidas:")
print(cols_fecha)

(8051, 88)
Dimensiones del archivo: (8051, 88)
Columnas de fecha convertidas:
['fecha_de_firma', 'fecha_de_inicio_del_contrato', 'fecha_de_fin_del_contrato', 'ultima_actualizacion', 'fecha_inicio_liquidacion', 'fecha_fin_liquidacion', 'fecha_de_notificacion_de_prorrogacion']


In [27]:
#Ingeniería de características

# Convertir la fecha de firma
df["fecha_firma"] = pd.to_datetime(
    df["fecha_de_firma"],
    errors="coerce"
)

# Crear el año de firma
df["anio_firma"] = df["fecha_firma"].dt.year

# Pasar el valor del contrato a millones de pesos
df["valor_millones"] = (
    df["valor_del_contrato"] / 1_000_000
)

# Organizar los nombres de los proveedores
df["proveedor"] = (
    df["proveedor_adjudicado"]
    .str.strip()
    .str.upper()
)

# Usar el código como identificador del proveedor
df["id_proveedor"] = (
    df["codigo_proveedor"].astype(str)
)

# Fecha de corte del proyecto
fecha_corte = pd.Timestamp("2026-08-24")

# Seleccionar contratos firmados, con valor positivo
# y dentro de la fecha de corte
df_analisis = df[
    (df["fecha_firma"].notna())
    & (df["fecha_firma"] <= fecha_corte)
    & (df["valor_del_contrato"] > 0)
].copy()

print("Registros originales:", len(df))
print("Contratos para analizar:", len(df_analisis))
print(
    "Registros excluidos:",
    len(df) - len(df_analisis)
)

Registros originales: 8051
Contratos para analizar: 7325
Registros excluidos: 726


In [28]:
# Resumen por proveedor

proveedores = df_analisis.groupby(
    ["id_proveedor", "proveedor"]
).agg(
    numero_contratos=("id_contrato", "count"),
    valor_total=("valor_del_contrato", "sum")
).reset_index()

# Ordenar de mayor a menor valor contratado
proveedores = proveedores.sort_values(
    "valor_total",
    ascending=False
).reset_index(drop=True)

# Convertir el valor a millones
proveedores["valor_total_millones"] = (
    proveedores["valor_total"] / 1_000_000
)

# Participación de cada proveedor
proveedores["participacion"] = (
    proveedores["valor_total"]
    / proveedores["valor_total"].sum()
)

# Participación acumulada
proveedores["participacion_acumulada"] = (
    proveedores["participacion"].cumsum()
)

# Crear el ranking
proveedores["ranking"] = (
    proveedores.index + 1
)

print(proveedores.head(10))

  id_proveedor                            proveedor  numero_contratos  \
0    704530401                            CORREAGRO                15   
1    711385625  UNION TEMPORAL GLOBAL ALLIANZ GROUP                 1   
2    728849431        RENTA Y CAMPO CORREDORES S.A.                 5   
3    704061126     COMISIONISTAS AGROPECUARIOS S.A.                10   
4    702669318       MIGUEL QUIJANO Y COMPAÑIA S.A.                 9   
5    702220401                     ALIMENTOS JACLER                10   
6    700811110                        FUEL SERVICES                13   
7    701922239                          INCOMSA SAS                 2   
8    701506016         ELKIN ALONSO TOBON CAMPUZANO                79   
9    701594459                            CODIS S.A                 4   

    valor_total  valor_total_millones  participacion  participacion_acumulada  \
0  6.865558e+11         686555.783550       0.129926                 0.129926   
1  3.983494e+11         398349.443

In [29]:
# Resumen por proveedor y modalidad

proveedores_modalidad = df_analisis.groupby(
    [
        "modalidad_de_contratacion",
        "id_proveedor",
        "proveedor"
    ]
).agg(
    numero_contratos=("id_contrato", "count"),
    valor_total=("valor_del_contrato", "sum")
).reset_index()

# Total contratado en cada modalidad
proveedores_modalidad["valor_modalidad"] = (
    proveedores_modalidad.groupby(
        "modalidad_de_contratacion"
    )["valor_total"].transform("sum")
)

# Participación del proveedor dentro de la modalidad
proveedores_modalidad["participacion_modalidad"] = (
    proveedores_modalidad["valor_total"]
    / proveedores_modalidad["valor_modalidad"]
)

# Posición del proveedor dentro de la modalidad
proveedores_modalidad["ranking_modalidad"] = (
    proveedores_modalidad.groupby(
        "modalidad_de_contratacion"
    )["valor_total"].rank(
        ascending=False,
        method="first"
    )
)

proveedores_modalidad = proveedores_modalidad.sort_values(
    [
        "modalidad_de_contratacion",
        "ranking_modalidad"
    ]
)

print("Filas:", len(proveedores_modalidad))
proveedores_modalidad.head(10)

Filas: 2377


,modalidad_de_contratacion,id_proveedor,proveedor,numero_contratos,valor_total,valor_modalidad,participacion_modalidad,ranking_modalidad
1,Concurso de méritos abierto,700189046,ERNST & YOUNG SAS,1,3.327597e+09,8.863317e+09,0.375435,1.0
4,Concurso de méritos abierto,701152142,AVANCE ORGANIZACIONAL CONSULTORES BIC,1,2.055000e+09,8.863317e+09,0.231854,2.0
12,Concurso de méritos abierto,704093129,CONSORCIO BASAN 2018,1,5.061377e+08,8.863317e+09,0.057105,3.0
2,Concurso de méritos abierto,700626013,CIVING INGENIEROS CONTRATISTAS SAS,1,4.381403e+08,8.863317e+09,0.049433,4.0
14,Concurso de méritos abierto,709143093,CONSORCIO DINÁMICO 2020,1,4.225999e+08,8.863317e+09,0.047680,5.0
15,Concurso de méritos abierto,723125084,CONSORCIO CANTÓN 2023,1,3.728675e+08,8.863317e+09,0.042069,6.0
0,Concurso de méritos abierto,700098064,IPV6 TECHNOLOGY S.A.S,1,3.720000e+08,8.863317e+09,0.041971,7.0
6,Concurso de méritos abierto,701551509,CONSORCIO SAN GABRIEL,1,3.246000e+08,8.863317e+09,0.036623,8.0
13,Concurso de méritos abierto,704793025,CONSORCIO ESTUDIO SISMICO,1,2.779218e+08,8.863317e+09,0.031356,9.0
11,Concurso de méritos abierto,703208520,CONSORCIO SANTA MARIA,1,1.506000e+08,8.863317e+09,0.016991,10.0


In [31]:
from pathlib import Path

carpeta_salida = Path(
    r"C:\Users\danie\OneDrive - Universidad de los Andes\8vo semestre\Análitica computacional\proyecto\proyecto1-secop-agencia-logistica\03-analisis-datos"
)

# Crear la carpeta si no existe
carpeta_salida.mkdir(
    parents=True,
    exist_ok=True
)

df_analisis.to_csv(
    carpeta_salida / "base_analitica.csv",
    index=False,
    encoding="utf-8-sig"
)

proveedores.to_csv(
    carpeta_salida / "resumen_proveedores.csv",
    index=False,
    encoding="utf-8-sig"
)

proveedores_modalidad.to_csv(
    carpeta_salida / "proveedores_modalidad.csv",
    index=False,
    encoding="utf-8-sig"
)

print("Archivos exportados en:")
print(carpeta_salida)

Archivos exportados en:
C:\Users\danie\OneDrive - Universidad de los Andes\8vo semestre\Análitica computacional\proyecto\proyecto1-secop-agencia-logistica\03-analisis-datos
